<a href="https://colab.research.google.com/github/smosharof/Resume.Walkthrough/blob/main/Generative_AI_Leadership_Cancer_Patients_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %%
!python -m venv myenv
!source myenv/bin/activate
!pip install pandas numpy langchain openai langchain-community
!pip install tiktoken
!pip install faiss-cpu
!pip install --upgrade openai

Error: Command '['/content/myenv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/bin/bash: line 1: myenv/bin/activate: No such file or directory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 62.6 MB/s eta 0:00:00


In [1]:

# %%
import pandas as pd
import numpy as np
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS  # In-memory vector store
import openai
import kagglehub
# --- 1. Load the Dataset (Directly from Uploaded File) ---
try:
    # Load the dataset from the uploaded file
    df = pd.read_csv("global_cancer_patients_2015_2024.csv")  # Change filename if needed
    print("\nDataset loaded successfully!")
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head().to_markdown(index=False, numalign="left", stralign="left"))
    print("\nDataFrame information:")
    print(df.info())

except FileNotFoundError:
    print("Error: 'global_cancer_patients_2015_2024.csv' not found. Please ensure the file is uploaded to Colab.")
    exit()  # Stop execution if the CSV is not found

except Exception as e:
    print(f"An error occurred during dataset loading: {e}")
    exit()



Dataset loaded successfully!

First 5 rows of the DataFrame:
| Patient_ID   | Age   | Gender   | Country_Region   | Year   | Genetic_Risk   | Air_Pollution   | Alcohol_Use   | Smoking   | Obesity_Level   | Cancer_Type   | Cancer_Stage   | Treatment_Cost_USD   | Survival_Years   | Target_Severity_Score   |
|:-------------|:------|:---------|:-----------------|:-------|:---------------|:----------------|:--------------|:----------|:----------------|:--------------|:---------------|:---------------------|:-----------------|:------------------------|
| PT0000000    | 71    | Male     | UK               | 2021   | 6.4            | 2.8             | 9.5           | 0.9       | 8.7             | Lung          | Stage III      | 62913.4              | 5.9              | 4.92                    |
| PT0000001    | 34    | Male     | China            | 2021   | 1.3            | 4.5             | 3.7           | 3.9       | 6.3             | Leukemia      | Stage 0        | 12573.4              |

In [3]:
# --- 2. Data Preparation for RAG ---
# Combine relevant columns into a single 'context' column (adjust as needed)
# This is crucial for providing the chatbot with information.

# Check if the column names are correct and if not adjust them accordingly. For instance, 'cancer type' instead of 'Cancer Type'.
df['context'] = df['Cancer_Type'].astype(str) + " " + df['Gender'].astype(str) + " " + df['Cancer_Stage'].astype(str) + " " + df['Treatment_Cost_USD'].astype(str) + " " + df['Survival_Years'].astype(str) + " " + df['Target_Severity_Score'].astype(str)

# Drop rows with missing 'context' (after combining)
df.dropna(subset=['context'], inplace=True)

# Text Splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Adjust as needed
    chunk_overlap=50   # Adjust as needed
)

documents = text_splitter.create_documents(df['context'].tolist())

# --- 3. Embedding and Vector Store ---
# Initialize OpenAI embeddings
openai_api_key = "Enter Key"  # Replace with your actual key
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

# Create the vector store
vectorstore = FAISS.from_documents(documents, embeddings)


In [15]:
import time
from tenacity import retry, wait_exponential, stop_after_attempt
import openai

# ... other imports ...

@retry(wait=wait_exponential(multiplier=1, min=1, max=60), stop=stop_after_attempt(6))
def rag_query(query, vectorstore, openai_api_key, model_name="text-davinci-003", max_tokens=150):
    """
    Performs a retrieval-augmented generation query.

    Args:
        query (str): The user's query.
        vectorstore: The FAISS vector store.
        openai_api_key (str): Your OpenAI API key.
        model_name (str): The OpenAI model to use.
        max_tokens (int): Maximum tokens in the response.

    Returns:
        str: The generated response.
    """

    # Retrieve relevant documents
    relevant_docs = vectorstore.similarity_search(query, k=3)  # Adjust 'k' as needed
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # Construct the prompt
    prompt = f"You are a helpful chatbot providing information about cancer patients. Use the following context to answer the user's question: \n\n{context}\n\nUser Query: {query}\n\n Response:"

    # Generate the response using OpenAI
    client = openai.OpenAI(api_key=openai_api_key) # initialize openai client

    # Updated to use the new API for chat models
    if "gpt-3.5-turbo" in model_name or "gpt-4" in model_name:  # Chat models
        # Use client.chat.completions.create for chat models
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        # Access the content from the response using the updated format
        return response.choices[0].message.content.strip()
    else:  # Completion models
        response = client.completions.create( # use client for completions
            model=model_name,
            prompt=prompt,
            max_tokens=max_tokens
        )
        return response.choices[0].text.strip()

In [17]:
# --- 5. Interactive Chat with the Chatbot ---
print("\n--- Interactive Chat with the Cancer Information Chatbot ---")
print("Type 'exit' to end the chat.")

while True:
    user_query = input("\nUser: ")
    if user_query.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    answer = rag_query(user_query, vectorstore, openai_api_key, model_name="gpt-3.5-turbo")  # Or "text-davinci-003"
    print(f"Chatbot: {answer}")



--- Interactive Chat with the Cancer Information Chatbot ---
Type 'exit' to end the chat.

User: Tell me about skin cance
Chatbot: Skin cancer is a type of cancer that begins in the skin cells. It is typically categorized into different stages, such as stage IV. The numbers mentioned in your query (62031.38, 28580.41, 6485.7) could potentially represent data related to skin cancer cases, such as tumor size, survival rates, or other specific information. It is important for individuals diagnosed with skin cancer to consult with a healthcare provider for personalized information and treatment options.

User: What are the symptoms of lung cancer
Chatbot: Some common symptoms of lung cancer include persistent cough, chest pain, shortness of breath, coughing up blood, hoarseness, unexplained weight loss, fatigue, and recurring respiratory infections. It is important to note that symptoms may vary depending on the stage and type of lung cancer. If you are experiencing any of these symptoms,